# 🌊 MEI.v2 (Multivariate ENSO Index Version 2) 데이터 개요

## 1. MEI.v2란 무엇인가?

단순히 바다 온도만 측정하는 일반적인 엘니뇨 지표(NINO3.4 등)와 달리, **해양과 대기 전반의 상태를 통합하여 분석하는 '하이브리드' 지수**입니다. NOAA(미국 해양대기청)에서 생산하며, 현재 엘니뇨/라니뇨의 강도와 진행 상태를 가장 입체적으로 보여주는 지표 중 하나로 평가받습니다.

* **데이터 소스:** NOAA PSL (Physical Sciences Laboratory)
* **업데이트:** 매월 초 (실시간 모니터링 가능)
* **분석 단위:** 월별 시계열

## 2. 왜 '다변수(Multivariate)'인가?

MEI.v2는 아래 **5가지 핵심 기상 변수**를 복합적으로 결합하여 산출됩니다. 단순히 뜨겁고 차가운 것을 넘어, 바람의 흐름과 기압의 변화까지 담고 있습니다.

* **해수면 온도 (SST):** 바다 표면의 열 상태.
* **해면 기압 (SLP):** 대기의 고기압/저기압 배치.
* **지표 바람 (U & V components):** 동서 및 남북 방향의 바람 세기.
* **방출 장파 복사 (OLR):** 구름의 양과 강수 활동 추정치.

## 3. 주요 특징 및 장점

- **분석 정밀도** : 해양-대기 결합(Coupling)을 반영하므로, 기온만 보는 지수보다 **기후 시스템의 급격한 변화를 더 민감하게 포착**합니다.
- **시간 해상도** : **2개월 중첩 시즌(Bimonthly Overlapping Seasons)** 방식을 사용합니다. (예: DJ, JF, FM...) 이는 계절적 변동성을 매끄럽게(Smoothing) 처리하여 노이즈를 줄여줍니다.
- **예측력** : 동아프리카처럼 대기 순환의 영향을 크게 받는 지역의 **식량 가격 및 수확량 예측 모델**에서 일반 SST 지수보다 높은 상관관계를 보이는 경우가 많습니다.

## 4. 수치 해석 방법

데이터프레임의 `MEI_v2` 값이 의미하는 바는 다음과 같습니다.

* **Positive (+) 값 (엘니뇨 경향):** 값이 클수록 강력한 엘니뇨를 의미합니다. 전 지구적 기온 상승과 연관되며, 인도양 다이폴(IOD)과 상호작용하여 동아프리카에 이상 기후를 유발합니다.
* **Negative (-) 값 (라니뇨 경향):** 값이 작을수록 강력한 라니뇨를 의미합니다. **에티오피아를 포함한 동아프리카 지역에 극심한 가뭄**을 일으키는 주범으로 작용하는 경우가 많습니다.
* **0 근처:** 중립(Neutral) 상태입니다.

In [1]:
import pandas as pd
import requests
import io

# 1. NOAA PSL MEI.v2 데이터 원천 URL
MEI_URL = "https://psl.noaa.gov/enso/mei/data/meiv2.data"

In [2]:
# 2. 데이터 가져오기
response = requests.get(MEI_URL)
if response.status_code != 200:
    print("❌ 다운로드 실패!")

In [3]:
# 3. 텍스트 파싱 (상단 헤더와 하단 푸터 제외)
# MEI 파일은 고정 폭(Fixed-width) 형식이지만 read_table로 처리가능합니다.
lines = response.text.split('\n')

# 실제 데이터가 시작되는 행과 끝나는 행 찾기 (연도 숫자로 시작하는 구간)
data_lines = []
for line in lines:
    parts = line.split()
    if len(parts) > 0 and parts[0].isdigit() and len(parts[0]) == 4:
        # 1979년부터 현재까지의 데이터만 추출
        data_lines.append(line)

In [4]:
# 데이터프레임 생성 (연도 + 12개 중첩 시즌)
columns = ['Year', 'DJ', 'JF', 'FM', 'MA', 'AM', 'MJ', 'JJ', 'JA', 'AS', 'SO', 'ON', 'ND']
df_raw = pd.read_csv(io.StringIO('\n'.join(data_lines[1:])), sep=r'\s+', names=columns)

# -999.00 같은 결측치 처리
df_raw = df_raw.replace(-999.00, pd.NA)

In [5]:
# 4. '연도-월' 형태의 Long-format으로 변환 (Melting)
# 식량 가격 데이터와 합치기 좋게 세로로 길게 늘립니다.
df_melted = df_raw.melt(id_vars=['Year'], var_name='Season', value_name='MEI_v2')

In [6]:
# 시즌 코드를 숫자로 매핑 (JF를 2월로 매핑하는 것이 일반적)
season_map = {
    'DJ': 1, 'JF': 2, 'FM': 3, 'MA': 4, 'AM': 5, 'MJ': 6,
    'JJ': 7, 'JA': 8, 'AS': 9, 'SO': 10, 'ON': 11, 'ND': 12
}
df_melted['Month'] = df_melted['Season'].map(season_map)

In [7]:
# 날짜 컬럼 생성 및 정렬
df_melted['date'] = pd.to_datetime(df_melted[['Year', 'Month']].assign(day=1))
df_final = df_melted[['date', 'Year', 'Month', 'MEI_v2']].sort_values('date').reset_index(drop=True)

In [8]:
# 우리가 분석하는 2007년 이후 데이터만 필터링
mei_df = df_final[(df_final['Year'] >= 2007) & (df_final['Year'] <= 2025)].dropna()

In [10]:
import os
from google.colab import drive

drive.mount('/content/drive')

# GEE 코드에서 설정한 폴더명과 파일명
folder_path = '/content/drive/MyDrive/GEE_Climate_Indices'
file_name = 'MEI_v2_Indices_2007_2025.csv'
full_path = os.path.join(folder_path, file_name)

Mounted at /content/drive


In [11]:
# 5. 결과 확인 및 저장
if mei_df is not None:
    print(mei_df.head())
    mei_df.to_csv(full_path, index=False)
    print("\n💾 MEI_v2_Indices_2007_2025.csv 저장 완료")

          date  Year  Month MEI_v2
336 2007-01-01  2007      1   0.64
337 2007-02-01  2007      2    0.4
338 2007-03-01  2007      3  -0.19
339 2007-04-01  2007      4  -0.32
340 2007-05-01  2007      5  -0.41

💾 MEI_v2_Indices_2007_2025.csv 저장 완료
